In [2]:
import pandas as pd

In [3]:
df = pd.read_parquet('mlsd-eda.parquet', engine='fastparquet')
df['date'] = df['event_date'].dt.date
df['hour'] = df['event_date'].dt.hour
df['day_num'] = df['event_date'].dt.dayofweek
df

,event_date,eid,x_eid,user_id,item_id,microcat_id,date,hour,day_num
0,2023-04-16 23:59:59,301,300,2010890,1435852750630,3674,2023-04-16,23,6
1,2023-04-16 23:59:59,301,300,379294501,1437733502707,1673,2023-04-16,23,6
2,2023-04-16 23:59:59,301,300,198428250055,1444440250530,3831,2023-04-16,23,6
3,2023-04-16 23:59:59,301,2012,705812500004,1444999750230,347447750001,2023-04-16,23,6
4,2023-04-16 23:59:59,301,2012,1116509750025,1378823251390,1804,2023-04-16,23,6
...,...,...,...,...,...,...,...,...,...
19083062,2023-04-10 00:00:10,301,300,988034750105,1096975250872,1381,2023-04-10,0,0
19083063,2023-04-10 00:00:10,301,2012,391780250051,1439281004214,299603250002,2023-04-10,0,0
19083064,2023-04-10 00:00:09,301,2437,284945250008,1433460000828,2105,2023-04-10,0,0
19083065,2023-04-10 00:00:08,301,400,1003548250074,1372700500868,503316,2023-04-10,0,0


In [4]:
df.columns

Index(['event_date', 'eid', 'x_eid', 'user_id', 'item_id', 'microcat_id',
       'date', 'hour', 'day_num'],
      dtype='object')

# Задачи

## Задание 1

Какое самое большое число различных объявлений (событие 401) пользователь добавил в избранное за день?

In [5]:
df_favorites = df[df['eid'] == 401]
favorites_per_day = df_favorites.groupby(['user_id', 'date'])['item_id'].nunique()
max(favorites_per_day)

833

## Задание 2

Какая доля объявлений имеет только просмотры (301 событие)? Напишите ответ от 0 до 1 с точностью до 2 знака после запятой.

In [6]:
n_item = df['item_id'].nunique()
(n_item - len(df[df['eid'] != 301]['item_id'].unique())) / n_item

0.6040478514698472

## Задание 3

В какой час пользователи больше всего делятся объявлениями в соцсетях (308 событие) ? В ответ напишите целое число от 0 до 23.

In [7]:
best_hour = df[df['eid'] == 308].groupby(['hour'])['x_eid'].count().idxmax()
print(best_hour)

17


## Задание 4

В какой день недели доля контактов (303 и 857) с главной страницы (2012) от всех контактов наибольшая?

In [8]:
main_page_contacts = df[(df['x_eid'] == 2012) & (df['eid'].isin([303, 857]))]
all_contacts = df[(df['eid'].isin([303, 857]))]

main_page_by_day = main_page_contacts.groupby(['day_num'])['eid'].count()
all_by_day = all_contacts.groupby(['day_num'])['eid'].count()

(main_page_by_day / all_by_day).idxmax()

np.int32(6)

## Задание 5

Сколько минут длилась самая длинная по времени сессия? Сессия - такая последовательность событий одного пользователя, что между двумя соседними по времени событиями разница не более 10 минут. Длина сессии - разность между первым и последним событием. Дайте ответ в минутах (секунды просто отбросить, не округлять).

In [9]:
sort_df = df.sort_values(['user_id', 'event_date'])
sort_df

,event_date,eid,x_eid,user_id,item_id,microcat_id,date,hour,day_num
16158814,2023-04-10 20:20:23,301,300,8,1387716002217,299603000002,2023-04-10,20,0
15906413,2023-04-11 00:26:36,301,300,8,1442728502422,338648500001,2023-04-11,0,1
15905735,2023-04-11 00:27:53,857,300,8,1442728502422,338648500001,2023-04-11,0,1
15905673,2023-04-11 00:28:02,401,300,8,1442728502422,338648500001,2023-04-11,0,1
15845429,2023-04-11 02:25:55,301,300,8,1091515500305,338654500001,2023-04-11,2,1
...,...,...,...,...,...,...,...,...,...
30419,2023-04-16 23:10:43,857,300,1196048000002,1435241751455,80720750004,2023-04-16,23,6
30381,2023-04-16 23:10:45,301,300,1196048000002,1435241751455,80720750004,2023-04-16,23,6
4881,2023-04-16 23:51:01,301,300,1196048500154,1443319752390,338649750001,2023-04-16,23,6
15885,2023-04-16 23:33:37,301,300,1196051000140,1438829500249,1507,2023-04-16,23,6


In [10]:
sort_df['ts_seconds'] = sort_df['event_date'].view('int64') // 10**9
sort_df['time_diff'] = sort_df.groupby('user_id')['ts_seconds'].diff()
sort_df[['user_id', 'event_date', 'time_diff']]

C:\Users\danii\AppData\Local\Temp\ipykernel_22260\1932673794.py:1: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  sort_df['ts_seconds'] = sort_df['event_date'].view('int64') // 10**9


,user_id,event_date,time_diff
16158814,8,2023-04-10 20:20:23,NaN
15906413,8,2023-04-11 00:26:36,14773.0
15905735,8,2023-04-11 00:27:53,77.0
15905673,8,2023-04-11 00:28:02,9.0
15845429,8,2023-04-11 02:25:55,7073.0
...,...,...,...
30419,1196048000002,2023-04-16 23:10:43,62.0
30381,1196048000002,2023-04-16 23:10:45,2.0
4881,1196048500154,2023-04-16 23:51:01,NaN
15885,1196051000140,2023-04-16 23:33:37,NaN


In [11]:
sort_df['is_new_session'] = (sort_df['time_diff'].fillna(9999) > 600).astype(int)
sort_df[['user_id', 'event_date', 'time_diff', 'is_new_session']]

,user_id,event_date,time_diff,is_new_session
16158814,8,2023-04-10 20:20:23,NaN,1
15906413,8,2023-04-11 00:26:36,14773.0,1
15905735,8,2023-04-11 00:27:53,77.0,0
15905673,8,2023-04-11 00:28:02,9.0,0
15845429,8,2023-04-11 02:25:55,7073.0,1
...,...,...,...,...
30419,1196048000002,2023-04-16 23:10:43,62.0,0
30381,1196048000002,2023-04-16 23:10:45,2.0,0
4881,1196048500154,2023-04-16 23:51:01,NaN,1
15885,1196051000140,2023-04-16 23:33:37,NaN,1


In [12]:
sort_df['session_id'] = sort_df.groupby('user_id')['is_new_session'].cumsum()
sort_df[['user_id', 'event_date', 'time_diff', 'session_id']]

,user_id,event_date,time_diff,session_id
16158814,8,2023-04-10 20:20:23,NaN,1
15906413,8,2023-04-11 00:26:36,14773.0,2
15905735,8,2023-04-11 00:27:53,77.0,2
15905673,8,2023-04-11 00:28:02,9.0,2
15845429,8,2023-04-11 02:25:55,7073.0,3
...,...,...,...,...
30419,1196048000002,2023-04-16 23:10:43,62.0,1
30381,1196048000002,2023-04-16 23:10:45,2.0,1
4881,1196048500154,2023-04-16 23:51:01,NaN,1
15885,1196051000140,2023-04-16 23:33:37,NaN,1


In [13]:
sessions_duration = sort_df.groupby(['user_id', 'session_id'])['event_date'].agg(['min', 'max'])
sessions_duration['diff'] = sessions_duration['max'] - sessions_duration['min']
max(sessions_duration['diff'])

Timedelta('0 days 05:08:15')

## Задание 6

Сколько пользователей чаще пользуются рекомендациями (источники действий - 2012 и 2437), чем поиском (300)? В ответ напишите целое число - количество пользователей.

In [14]:
rec_df = df[(df['x_eid'] == 2012) | (df['x_eid'] == 2437)]
find_df = df[df['x_eid'] == 300]

In [15]:
rec_counts = rec_df.groupby(['user_id'])['x_eid'].count()
find_counts = find_df.groupby(['user_id'])['x_eid'].count()

In [16]:
comparison_df = pd.concat([rec_counts, find_counts], axis=1)
comparison_df.columns = ['rec', 'search']
comparison_df = comparison_df.fillna(0)
comparison_df['diff'] = comparison_df['rec'] > comparison_df['search']
comparison_df['diff'].sum()

np.int64(750925)

## Задание 7

Какая микрокатегория имеет наибольшее среднее количество действий на объявление? Рассматриваем только микрокатегории, где более 100 уникальных объявлений. В ответ запишите microcat_id.

In [45]:
df_cat = df.groupby(['microcat_id'])['item_id'].nunique() > 100
popular_cats = df_cat[df_cat].index

In [46]:
df_pop_cat = df[df['microcat_id'].isin(popular_cats)]

In [47]:
item_actions = df_pop_cat.groupby(['microcat_id', 'item_id'])['x_eid'].count()

In [48]:
final_stats = item_actions.groupby(level=0).mean()
final_stats.idxmax()

np.int64(186)

## Задание 8

In [65]:
item_counts = df['item_id'].value_counts()

top_item_id = item_counts.idxmax()
top_item_actions = item_counts.max()

In [ ]:
top_item_cat = df[df['item_id'] == top_item_id]['microcat_id'].iloc[0]

In [ ]:
total_cat_actions = df[df['microcat_id'] == top_item_cat].shape[0]

share = top_item_actions / total_cat_actions
print(round(share, 2))

0.92


## Задание 9

In [61]:
users_259 = df[df['microcat_id'] == 259]['user_id'].unique()

In [62]:
df_subset = df[df['user_id'].isin(users_259)]

In [64]:
df_others = df_subset[df_subset['microcat_id'] != 259]
similarity_scores = df_others.groupby('microcat_id')['user_id'].nunique()
result_id = similarity_scores.idxmax()
result_id

np.int64(6702)

## Задание 10

Из какого города данные?

In [57]:
df.groupby('hour')['eid'].count()

hour
0      187317
1      179374
2      308678
3      503507
4      686503
5      809645
6      894847
7      950578
8      986461
9      993513
10    1009158
11    1009059
12    1006072
13    1015509
14    1044756
15    1095128
16    1122350
17    1235241
18    1244424
19    1045654
20     719426
21     465431
22     319796
23     250640
Name: eid, dtype: int64

In [60]:
hourly_activity = df.groupby('hour')['x_eid'].count()
print(hourly_activity)

print(f"Самый тихий час: {hourly_activity.idxmin()}")

hour
0      187317
1      179374
2      308678
3      503507
4      686503
5      809645
6      894847
7      950578
8      986461
9      993513
10    1009158
11    1009059
12    1006072
13    1015509
14    1044756
15    1095128
16    1122350
17    1235241
18    1244424
19    1045654
20     719426
21     465431
22     319796
23     250640
Name: x_eid, dtype: int64
Самый тихий час: 1
